<a href="https://colab.research.google.com/github/mebre7/house-price-predictor-mlops/blob/colab_branch/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering
Feature engineering is the process of using domain knowledge to create new features from existing data. It involves transforming raw data into a format that is more suitable for machine learning algorithms, which can lead to improved model performance.
- In this notebook of the house price predictor project, I will be performing feature engineering on the dataset to create new features that can help improve the predictive power of the model. This may include creating interaction terms, polynomial features, or aggregating existing features to capture more complex relationships in the data.

## Data loading

In [295]:
from sqlalchemy import create_engine, text
import pandas as pd
from urllib.parse import quote_plus
from google.colab import userdata

In [296]:
DB_USER = userdata.get("SUPABASE_USER")
DB_PASSWORD = quote_plus(userdata.get("SUPABASE_PASSWORD"))
DB_HOST = userdata.get("SUPABASE_HOST")
DB_PORT = userdata.get("SUPABASE_PORT")
DB_NAME = userdata.get("SUPABASE_NAME")

# 2. Build SQLAlchemy connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
with engine.connect() as conn:
    df = pd.read_sql(
        text("SELECT * FROM housing_data"),
        conn
    )

In [297]:
df.head()

,Order,PID,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,None,IR1,Lvl,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,None,Reg,Lvl,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,None,IR1,Lvl,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,None,Reg,Lvl,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,None,IR1,Lvl,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


In [298]:
df = df.drop(columns=["Order", "PID"])

In [299]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,141.0,31770,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,None,Reg,Lvl,AllPub,Inside,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,None,Reg,Lvl,AllPub,Corner,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,None,IR1,Lvl,AllPub,Inside,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


## Import Packages

In [300]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

## Features classification

**1. Numerical features**

In [301]:
# define numerical features
numerical_features_all = df.select_dtypes(include=[np.number]).columns.tolist()

In [302]:
# Discrete features
discrete_num_features = ['MS_SubClass', 'Overall_Qual', 'Overall_Cond', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Cars']
discrete_num_features

['MS_SubClass',
 'Overall_Qual',
 'Overall_Cond',
 'Bsmt_Full_Bath',
 'Bsmt_Half_Bath',
 'Full_Bath',
 'Half_Bath',
 'Bedroom_AbvGr',
 'Kitchen_AbvGr',
 'TotRms_AbvGrd',
 'Fireplaces',
 'Garage_Cars']

In [303]:
# Continuous features
continuous_num_features = list(set(numerical_features_all) - set(discrete_num_features))
continuous_num_features

['1st_Flr_SF',
 '3Ssn_Porch',
 'SalePrice',
 'Wood_Deck_SF',
 'Mas_Vnr_Area',
 'Gr_Liv_Area',
 'Lot_Area',
 'BsmtFin_SF_2',
 'Yr_Sold',
 'Screen_Porch',
 'Mo_Sold',
 'Misc_Val',
 'Low_Qual_Fin_SF',
 '2nd_Flr_SF',
 'Total_Bsmt_SF',
 'Enclosed_Porch',
 'Year_Built',
 'BsmtFin_SF_1',
 'Year_Remod/Add',
 'Pool_Area',
 'Lot_Frontage',
 'Garage_Area',
 'Open_Porch_SF',
 'Bsmt_Unf_SF',
 'Garage_Yr_Blt']

In [304]:
num_feats_int = list(set(numerical_features_all) - set(discrete_num_features)) + ['SalePrice']

**2. Categorical features**

In [305]:
# define categorical features
categorical_features = df.select_dtypes(include=[object]).columns.tolist()

In [306]:
# Add discrete features to categorical features list
categorical_features_all = list(set(categorical_features + discrete_num_features))

In [307]:
categorical_features_all

['Paved_Drive',
 'Utilities',
 'House_Style',
 'Roof_Matl',
 'Foundation',
 'Garage_Type',
 'Half_Bath',
 'Bsmt_Half_Bath',
 'Fireplace_Qu',
 'MS_SubClass',
 'TotRms_AbvGrd',
 'Bsmt_Full_Bath',
 'Sale_Type',
 'Kitchen_AbvGr',
 'Garage_Cond',
 'Bldg_Type',
 'Condition_2',
 'Bedroom_AbvGr',
 'Central_Air',
 'Overall_Cond',
 'Bsmt_Qual',
 'Garage_Qual',
 'Bsmt_Exposure',
 'MS_Zoning',
 'Mas_Vnr_Type',
 'BsmtFin_Type_1',
 'Exter_Qual',
 'Full_Bath',
 'Condition_1',
 'Exter_Cond',
 'Bsmt_Cond',
 'BsmtFin_Type_2',
 'Electrical',
 'Land_Slope',
 'Pool_QC',
 'Garage_Finish',
 'Overall_Qual',
 'Functional',
 'Heating',
 'Lot_Shape',
 'Alley',
 'Exterior_1st',
 'Lot_Config',
 'Kitchen_Qual',
 'Neighborhood',
 'Exterior_2nd',
 'Street',
 'Garage_Cars',
 'Heating_QC',
 'Fireplaces',
 'Sale_Condition',
 'Fence',
 'Land_Contour',
 'Misc_Feature',
 'Roof_Style']

**Change data types of `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [308]:
for col in categorical_features_all:
    df[col] = df[col].astype('category')

In [309]:
df[categorical_features_all].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 55 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   Paved_Drive     2930 non-null   category
 1   Utilities       2930 non-null   category
 2   House_Style     2930 non-null   category
 3   Roof_Matl       2930 non-null   category
 4   Foundation      2930 non-null   category
 5   Garage_Type     2773 non-null   category
 6   Half_Bath       2930 non-null   category
 7   Bsmt_Half_Bath  2928 non-null   category
 8   Fireplace_Qu    1508 non-null   category
 9   MS_SubClass     2930 non-null   category
 10  TotRms_AbvGrd   2930 non-null   category
 11  Bsmt_Full_Bath  2928 non-null   category
 12  Sale_Type       2930 non-null   category
 13  Kitchen_AbvGr   2930 non-null   category
 14  Garage_Cond     2771 non-null   category
 15  Bldg_Type       2930 non-null   category
 16  Condition_2     2930 non-null   category
 17  Bedroom_AbvGr 

**3. Temporal features**

In [310]:
date_year = ['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', ]
date_month = ['Mo_Sold']

for col in date_year:
    df[col] = pd.to_datetime(df[col], format='%Y').dt.year

for col in date_month:
    df[col] = pd.to_datetime(df[col], format='%m').dt.month

In [311]:
# Temporal features
temporal_features = date_year + date_month
temporal_features

['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', 'Mo_Sold']

**Summarize**

In [312]:

# print columns
print(f'{len(numerical_features_all)} numerical features: {numerical_features_all}')
print(f'\n{len(categorical_features)} categorical features: {categorical_features}')
print(f'\n{len(temporal_features)} temporal features: {temporal_features}')

37 numerical features: ['MS_SubClass', 'Lot_Frontage', 'Lot_Area', 'Overall_Qual', 'Overall_Cond', 'Year_Built', 'Year_Remod/Add', 'Mas_Vnr_Area', 'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF', '1st_Flr_SF', '2nd_Flr_SF', 'Low_Qual_Fin_SF', 'Gr_Liv_Area', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Yr_Blt', 'Garage_Cars', 'Garage_Area', 'Wood_Deck_SF', 'Open_Porch_SF', 'Enclosed_Porch', '3Ssn_Porch', 'Screen_Porch', 'Pool_Area', 'Misc_Val', 'Mo_Sold', 'Yr_Sold', 'SalePrice']

43 categorical features: ['MS_Zoning', 'Street', 'Alley', 'Lot_Shape', 'Land_Contour', 'Utilities', 'Lot_Config', 'Land_Slope', 'Neighborhood', 'Condition_1', 'Condition_2', 'Bldg_Type', 'House_Style', 'Roof_Style', 'Roof_Matl', 'Exterior_1st', 'Exterior_2nd', 'Mas_Vnr_Type', 'Exter_Qual', 'Exter_Cond', 'Foundation', 'Bsmt_Qual', 'Bsmt_Cond', 'Bsmt_Exposure', 'BsmtFin_Type_1', 'BsmtFin_Type_2', 'Heating', 'H

In [313]:
# total
print(f'Total features: {len(numerical_features_all) + len(categorical_features)}')

Total features: 80


## 1. Outlier Removal/Resolving

Important concepts in outlier detection! Here's a breakdown:

*   **Interquartile Range (IQR):** The IQR is a measure of statistical dispersion, representing the range of the middle 50% of your data. It's calculated as the difference between the third quartile (Q3) and the first quartile (Q1).

    *   **Q1 (First Quartile):** The value below which 25% of the data falls.
    *   **Q3 (Third Quartile):** The value below which 75% of the data falls.
    *   `IQR = Q3 - Q1`

*   **Outlier Thresholds (Lower Bound and Upper Bound):** These are calculated boundaries used to identify potential outliers in a dataset. They are derived using the IQR.

    *   **Lower Bound:** `Q1 - 1.5 * IQR`
    *   **Upper Bound:** `Q3 + 1.5 * IQR`

*   **Outlier vs. IQR:**
    *   The **IQR** itself is a measure of the spread of the central portion of the data, robust to extreme values.
    *   **Outliers** are data points that fall outside the calculated lower and upper bounds. The 1.5 multiplier in the threshold calculation is a commonly used convention (often referred to as the "1.5 IQR rule") to define what constitutes an outlier relative to the bulk of the data's spread. Any data point less than the lower bound or greater than the upper bound is considered an outlier.

In [314]:
num_feats_int

['1st_Flr_SF',
 '3Ssn_Porch',
 'SalePrice',
 'Wood_Deck_SF',
 'Mas_Vnr_Area',
 'Gr_Liv_Area',
 'Lot_Area',
 'BsmtFin_SF_2',
 'Yr_Sold',
 'Screen_Porch',
 'Mo_Sold',
 'Misc_Val',
 'Low_Qual_Fin_SF',
 '2nd_Flr_SF',
 'Total_Bsmt_SF',
 'Enclosed_Porch',
 'Year_Built',
 'BsmtFin_SF_1',
 'Year_Remod/Add',
 'Pool_Area',
 'Lot_Frontage',
 'Garage_Area',
 'Open_Porch_SF',
 'Bsmt_Unf_SF',
 'Garage_Yr_Blt',
 'SalePrice']

In [315]:
# Calculate outlier thresholds
def outlier_thresholds(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    #outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return lower_bound, upper_bound

In [316]:
# detect outliers
def detect_outliers(df, column):
    if pd.api.types.is_numeric_dtype(df[column]):
        low, up = outlier_thresholds(df, column)
        return (df[column] > up) | (df[column] < low)
    else:
        return pd.Series(False, index=df.index)

In [317]:
outliers_df = pd.DataFrame(index=df.index)
for col in numerical_features_all:
    outliers_df[col] = detect_outliers(df, col)

# Display the count of outliers per column
print("Number of outliers per numerical column:")
print(outliers_df.sum())

Number of outliers per numerical column:
MS_SubClass          0
Lot_Frontage       187
Lot_Area           127
Overall_Qual         0
Overall_Cond         0
Year_Built           9
Year_Remod/Add       0
Mas_Vnr_Area       200
BsmtFin_SF_1        15
BsmtFin_SF_2       351
Bsmt_Unf_SF         56
Total_Bsmt_SF      123
1st_Flr_SF          43
2nd_Flr_SF           8
Low_Qual_Fin_SF     40
Gr_Liv_Area         75
Bsmt_Full_Bath       0
Bsmt_Half_Bath       0
Full_Bath            0
Half_Bath            0
Bedroom_AbvGr        0
Kitchen_AbvGr        0
TotRms_AbvGrd        0
Fireplaces           0
Garage_Yr_Blt        3
Garage_Cars          0
Garage_Area         42
Wood_Deck_SF        67
Open_Porch_SF      159
Enclosed_Porch     459
3Ssn_Porch          37
Screen_Porch       256
Pool_Area           13
Misc_Val           103
Mo_Sold              0
Yr_Sold              0
SalePrice          137
dtype: int64


In [318]:
len(numerical_features_all)

37

In [319]:
len(discrete_num_features)

12

In [320]:
len(num_feats_int)

26

> $26 = 37 - 12 + 1$

In [321]:
display(outliers_df.head())

,MS_SubClass,Lot_Frontage,Lot_Area,Overall_Qual,Overall_Cond,Year_Built,Year_Remod/Add,Mas_Vnr_Area,BsmtFin_SF_1,BsmtFin_SF_2,...,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,3Ssn_Porch,Screen_Porch,Pool_Area,Misc_Val,Mo_Sold,Yr_Sold,SalePrice
0,False,True,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [322]:
# Function to check all numeric columns for outliers
def inspect_outliers(df, col_list):
    results = {}
    for col in col_list:
        results[col] = detect_outliers(df, col).any() if pd.api.types.is_numeric_dtype(df[col]) else False
    return results

In [323]:
# Function to display columns with outlier information
def display_outlier_info(df, col_list):
    print("Outlier thresholds for the numeric column")
    for col in col_list:
        low, up = outlier_thresholds(df, col)
        print(f"{col} : Lower Bound: {low}, Upper Bound: {up}", end="\n")
    print(end="\n\n")
    outliers_absent = []
    outliers_present = []
    for col, value in inspect_outliers(df, col_list).items():
        if value:
            outliers_present.append(col)
        else:
            outliers_absent.append(col)
    print("Columns with outliers")
    print(outliers_present)
    print(f"column count: {len(outliers_present)}", end="\n\n")
    print("Columns with no outliers")
    print(outliers_absent)
    print(f"column count: {len(outliers_absent)}", end="\n\n")

    return outliers_present, outliers_absent

In [324]:
# Resolve Outliers
def threshold_resolution(df, column):
    low, up = outlier_thresholds(df, column)
    df.loc[df[column] < low, column] = low
    df.loc[df[column] > up, column] = up
    return low, up

In [325]:
current_numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()
outliers_present, outliers_absent = display_outlier_info(df, current_numerical_columns)

Outlier thresholds for the numeric column
Lot_Frontage : Lower Bound: 25.0, Upper Bound: 113.0
Lot_Area : Lower Bound: 1267.75, Upper Bound: 17727.75
Year_Built : Lower Bound: 1883.5, Upper Bound: 2071.5
Year_Remod/Add : Lower Bound: 1906.5, Upper Bound: 2062.5
Mas_Vnr_Area : Lower Bound: -246.0, Upper Bound: 410.0
BsmtFin_SF_1 : Lower Bound: -1101.0, Upper Bound: 1835.0
BsmtFin_SF_2 : Lower Bound: 0.0, Upper Bound: 0.0
Bsmt_Unf_SF : Lower Bound: -655.5, Upper Bound: 1676.5
Total_Bsmt_SF : Lower Bound: 29.5, Upper Bound: 2065.5
1st_Flr_SF : Lower Bound: 114.625, Upper Bound: 2145.625
2nd_Flr_SF : Lower Bound: -1055.625, Upper Bound: 1759.375
Low_Qual_Fin_SF : Lower Bound: 0.0, Upper Bound: 0.0
Gr_Liv_Area : Lower Bound: 200.875, Upper Bound: 2667.875
Garage_Yr_Blt : Lower Bound: 1897.0, Upper Bound: 2065.0
Garage_Area : Lower Bound: -64.0, Upper Bound: 960.0
Wood_Deck_SF : Lower Bound: -252.0, Upper Bound: 420.0
Open_Porch_SF : Lower Bound: -105.0, Upper Bound: 175.0
Enclosed_Porch : L

**Now remove outliers**

In [326]:
for col in outliers_present:
    if col not in ['SalePrice']:
        threshold_resolution(df, col)

In [327]:
outliers_present, outliers_absent = display_outlier_info(df, current_numerical_columns)

Outlier thresholds for the numeric column
Lot_Frontage : Lower Bound: 25.0, Upper Bound: 113.0
Lot_Area : Lower Bound: 1267.75, Upper Bound: 17727.75
Year_Built : Lower Bound: 1883.5, Upper Bound: 2071.5
Year_Remod/Add : Lower Bound: 1906.5, Upper Bound: 2062.5
Mas_Vnr_Area : Lower Bound: -246.0, Upper Bound: 410.0
BsmtFin_SF_1 : Lower Bound: -1101.0, Upper Bound: 1835.0
BsmtFin_SF_2 : Lower Bound: 0.0, Upper Bound: 0.0
Bsmt_Unf_SF : Lower Bound: -655.5, Upper Bound: 1676.5
Total_Bsmt_SF : Lower Bound: 29.5, Upper Bound: 2065.5
1st_Flr_SF : Lower Bound: 114.625, Upper Bound: 2145.625
2nd_Flr_SF : Lower Bound: -1055.625, Upper Bound: 1759.375
Low_Qual_Fin_SF : Lower Bound: 0.0, Upper Bound: 0.0
Gr_Liv_Area : Lower Bound: 200.875, Upper Bound: 2667.875
Garage_Yr_Blt : Lower Bound: 1897.0, Upper Bound: 2065.0
Garage_Area : Lower Bound: -64.0, Upper Bound: 960.0
Wood_Deck_SF : Lower Bound: -252.0, Upper Bound: 420.0
Open_Porch_SF : Lower Bound: -105.0, Upper Bound: 175.0
Enclosed_Porch : L

> **Now no columns with outliers except `SalePrice`**

## 2. Train-Test split

In [328]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [329]:
X_train.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition
381,20,RL,80.0,10400.00,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,NaN,0,6,2009,WD,Family
834,60,RL,NaN,17727.75,Pave,NaN,IR2,Low,AllPub,CulDSac,...,0,0,NaN,NaN,NaN,0,6,2009,WD,Abnorml
1898,20,RL,75.0,7388.00,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,7,2007,WD,Normal
678,90,RL,60.0,8544.00,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,6,2009,WD,Normal
700,70,RM,57.0,6406.00,Pave,Grvl,Reg,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,NaN,0,10,2009,WD,Normal


In [330]:
X_test.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition
1357,50,RL,60.0,9144.0,Pave,Pave,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,3,2008,WD,Normal
2367,160,RM,25.0,1953.0,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
2822,20,RL,73.0,16133.0,Pave,NaN,Reg,HLS,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,12,2006,WD,Abnorml
2126,20,RL,70.0,16561.0,Pave,NaN,IR2,Low,AllPub,Inside,...,0,0,NaN,GdPrv,NaN,0,7,2007,WD,Normal
1544,160,RM,25.0,1950.0,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,GdPrv,NaN,0,7,2008,COD,Normal


In [331]:
y_train.head()

,SalePrice
381,152000
834,185000
1898,150750
678,92900
700,150000


In [332]:
y_test.head()

,SalePrice
1357,162500
2367,83000
2822,119900
2126,147900
1544,151000


## 3. Missing Value Handling (Imputation)

In [333]:
count = df.isnull().sum().sort_values(ascending=False)
percentage = ((df.isnull().sum() / df.isnull().count()) * 100).sort_values(ascending=False)
null_values = pd.concat([count, percentage], axis=1, keys=['Count', 'Percentage'])

In [334]:
null_values.head()

,Count,Percentage
Pool_QC,2917,99.556314
Misc_Feature,2824,96.382253
Alley,2732,93.242321
Fence,2358,80.477816
Mas_Vnr_Type,1775,60.580205


In [335]:
df.columns.to_list()

['MS_SubClass',
 'MS_Zoning',
 'Lot_Frontage',
 'Lot_Area',
 'Street',
 'Alley',
 'Lot_Shape',
 'Land_Contour',
 'Utilities',
 'Lot_Config',
 'Land_Slope',
 'Neighborhood',
 'Condition_1',
 'Condition_2',
 'Bldg_Type',
 'House_Style',
 'Overall_Qual',
 'Overall_Cond',
 'Year_Built',
 'Year_Remod/Add',
 'Roof_Style',
 'Roof_Matl',
 'Exterior_1st',
 'Exterior_2nd',
 'Mas_Vnr_Type',
 'Mas_Vnr_Area',
 'Exter_Qual',
 'Exter_Cond',
 'Foundation',
 'Bsmt_Qual',
 'Bsmt_Cond',
 'Bsmt_Exposure',
 'BsmtFin_Type_1',
 'BsmtFin_SF_1',
 'BsmtFin_Type_2',
 'BsmtFin_SF_2',
 'Bsmt_Unf_SF',
 'Total_Bsmt_SF',
 'Heating',
 'Heating_QC',
 'Central_Air',
 'Electrical',
 '1st_Flr_SF',
 '2nd_Flr_SF',
 'Low_Qual_Fin_SF',
 'Gr_Liv_Area',
 'Bsmt_Full_Bath',
 'Bsmt_Half_Bath',
 'Full_Bath',
 'Half_Bath',
 'Bedroom_AbvGr',
 'Kitchen_AbvGr',
 'Kitchen_Qual',
 'TotRms_AbvGrd',
 'Functional',
 'Fireplaces',
 'Fireplace_Qu',
 'Garage_Type',
 'Garage_Yr_Blt',
 'Garage_Finish',
 'Garage_Cars',
 'Garage_Area',
 'Garage_Qu

**Visualize missing values**

In [336]:
fig = px.bar(x=null_values.index, y=null_values['Percentage'],
             labels={'x': 'Features', 'y': 'Percent of missing values'},
             title='Percent missing data by feature')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**1. Structural Absence (Replace `NaN` with `None` or `0`)**

These columns use missing values to denote that the feature does not exist on the property.

- **`structural_categorical` (Fill with `None`)**

In [337]:
structural_categorical = [
    'Alley', 'Mas_Vnr_Type', 'Bsmt_Qual', 'Bsmt_Cond', 'Bsmt_Exposure',
    'BsmtFin_Type_1', 'BsmtFin_Type_2', 'Fireplace_Qu', 'Garage_Type',
    'Garage_Finish', 'Garage_Qual', 'Garage_Cond', 'Pool_QC', 'Fence', 'Misc_Feature'
]

```
df.Alley=["No Alley" if x is np.nan else x for x in df.Alley]
df.Mas_Vnr_Type=["Missing" if x is np.nan else x for x in df.Mas_Vnr_Type]
df.Bsmt_Qual=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Qual]
df.Bsmt_Cond=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Cond]
df.Bsmt_Exposure=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Exposure]
df.BsmtFin_Type_1=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_1]
df.BsmtFin_Type_2=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_2]
df.Fireplace_Qu=["No Firepl" if x is np.nan else x for x in df.Fireplace_Qu]
df.Garage_Type=["No Grg" if x is np.nan else x for x in df.Garage_Type]
df.Garage_Finish=["No Grg" if x is np.nan else x for x in df.Garage_Finish]
df.Garage_Qual=["No Grg" if x is np.nan else x for x in df.Garage_Qual]
df.Garage_Cond=["No Grg" if x is np.nan else x for x in df.Garage_Cond]
df.Pool_QC=["No Pool" if x is np.nan else x for x in df.Pool_QC]
df.Fence=["No Fence" if x is np.nan else x for x in df.Fence]
df.Misc_Feature=["None" if x is np.nan else x for x in df.Misc_Feature]
```

```py
df['Alley'] = df['Alley'].fillna('No Alley')
df['Mas_Vnr_Type'] = df['Mas_Vnr_Type'].fillna('Missing')
df['Bsmt_Qual'] = df['Bsmt_Qual'].fillna('No Bsmnt')
df['Bsmt_Cond'] = df['Bsmt_Cond'].fillna('No Bsmnt')
df['Bsmt_Exposure'] = df['Bsmt_Exposure'].fillna('No Bsmnt')
df['BsmtFin_Type_1'] = df['BsmtFin_Type_1'].fillna('No Bsmnt')
df['BsmtFin_Type_2'] = df['BsmtFin_Type_2'].fillna('No Bsmnt')
df['Fireplace_Qu'] = df['Fireplace_Qu'].fillna('No Firepl')
df['Garage_Type'] = df['Garage_Type'].fillna('No Grg')
df['Garage_Finish'] = df['Garage_Finish'].fillna('No Grg')
df['Garage_Qual'] = df['Garage_Qual'].fillna('No Grg')
df['Garage_Cond'] = df['Garage_Cond'].fillna('No Grg')
df['Pool_QC'] = df['Pool_QC'].fillna('No Pool')
df['Fence'] = df['Fence'].fillna('No Fence')
df['Misc_Feature'] = df['Misc_Feature'].fillna('None')
```

In [338]:
fill_values = {
    'Alley': 'No Alley',
    'Mas_Vnr_Type': 'Missing',
    'Bsmt_Qual': 'No Bsmnt',
    'Bsmt_Cond': 'No Bsmnt',
    'Bsmt_Exposure': 'No Bsmnt',
    'BsmtFin_Type_1': 'No Bsmnt',
    'BsmtFin_Type_2': 'No Bsmnt',
    'Fireplace_Qu': 'No Firepl',
    'Garage_Type': 'No Grg',
    'Garage_Finish': 'No Grg',
    'Garage_Qual': 'No Grg',
    'Garage_Cond': 'No Grg',
    'Pool_QC': 'No Pool',
    'Fence': 'No Fence',
    'Misc_Feature': 'None'
}

for col, fill_value in fill_values.items():
    if fill_value not in df[col].cat.categories:
        df[col] = df[col].cat.add_categories(fill_value)
    df[col] = df[col].fillna(fill_value)

In [339]:
df[structural_categorical].isnull().sum()

,0
Alley,0
Mas_Vnr_Type,0
Bsmt_Qual,0
Bsmt_Cond,0
Bsmt_Exposure,0
BsmtFin_Type_1,0
BsmtFin_Type_2,0
Fireplace_Qu,0
Garage_Type,0
Garage_Finish,0


- **`structural_numerical` (Fill with `0`)**

In [340]:
structural_numerical = [
    'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF',
    'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Garage_Cars', 'Garage_Area', 'Mas_Vnr_Area'
]

In [341]:
for col in structural_numerical:
    df[col] = df[col].fillna(0)

In [342]:
df[structural_numerical].isnull().sum()

,0
BsmtFin_SF_1,0
BsmtFin_SF_2,0
Bsmt_Unf_SF,0
Total_Bsmt_SF,0
Bsmt_Full_Bath,0
Bsmt_Half_Bath,0
Garage_Cars,0
Garage_Area,0
Mas_Vnr_Area,0


**Checking**

In [343]:
df[structural_numerical + structural_categorical].isnull().sum().sum()

np.int64(0)

In [344]:
len(structural_numerical + structural_categorical)

24

In [345]:
features_with_missing_values = list(set(df.columns.to_list()) - set(structural_numerical + structural_categorical))
print(len(features_with_missing_values))
features_with_missing_values

56


['Half_Bath',
 'Roof_Matl',
 'House_Style',
 'Paved_Drive',
 'Utilities',
 'Foundation',
 '1st_Flr_SF',
 'TotRms_AbvGrd',
 'MS_SubClass',
 'SalePrice',
 '3Ssn_Porch',
 'Wood_Deck_SF',
 'Sale_Type',
 'Kitchen_AbvGr',
 'Bldg_Type',
 'Bedroom_AbvGr',
 'Condition_2',
 'Central_Air',
 'Overall_Cond',
 'Gr_Liv_Area',
 'Lot_Area',
 'Yr_Sold',
 'MS_Zoning',
 'Screen_Porch',
 'Mo_Sold',
 'Exter_Qual',
 'Full_Bath',
 'Low_Qual_Fin_SF',
 '2nd_Flr_SF',
 'Misc_Val',
 'Condition_1',
 'Exter_Cond',
 'Enclosed_Porch',
 'Year_Built',
 'Electrical',
 'Land_Slope',
 'Year_Remod/Add',
 'Overall_Qual',
 'Functional',
 'Pool_Area',
 'Heating',
 'Lot_Shape',
 'Exterior_1st',
 'Lot_Config',
 'Lot_Frontage',
 'Kitchen_Qual',
 'Neighborhood',
 'Exterior_2nd',
 'Street',
 'Heating_QC',
 'Open_Porch_SF',
 'Fireplaces',
 'Sale_Condition',
 'Land_Contour',
 'Roof_Style',
 'Garage_Yr_Blt']

> Out of $80$ total features, $23$ features with missing values are handled considering _structuring absence_. Remaining $57$ will be handled by _statistical imputation_.

In [346]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,113.0,17727.75,Pave,No Alley,IR1,Lvl,AllPub,Corner,...,0,No Pool,No Fence,None,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622.00,Pave,No Alley,Reg,Lvl,AllPub,Inside,...,0,No Pool,MnPrv,None,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267.00,Pave,No Alley,IR1,Lvl,AllPub,Corner,...,0,No Pool,No Fence,Gar2,0,6,2010,WD,Normal,172000
3,20,RL,93.0,11160.00,Pave,No Alley,Reg,Lvl,AllPub,Corner,...,0,No Pool,No Fence,None,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830.00,Pave,No Alley,IR1,Lvl,AllPub,Inside,...,0,No Pool,MnPrv,None,0,3,2010,WD,Normal,189900


**2. Statistical Imputation (True Missingness/Clerical Errors)**

These features are physically required for a house to exist. Missing data here represents a processing omission or clerical typo.

- **`spatial_numerical` (Impute using the median of their specific `Neighborhood`)**

In [347]:
spatial_numerical = ['Lot_Frontage']

In [348]:
df[spatial_numerical].isnull().sum()

,0
Lot_Frontage,490


In [349]:
df['Lot_Frontage'].interpolate(method='linear', axis=0, inplace=True)

In [350]:
df[spatial_numerical].isnull().sum()

,0
Lot_Frontage,0


- **`mode_categorical` (Impute using the column's global mode/most frequent item)**

In [351]:
mode_categorical = [
    'Utilities', 'Exterior_1st', 'Exterior_2nd', 'Electrical',
    'Kitchen_Qual', 'Functional', 'Sale_Type'
]

In [352]:
df[mode_categorical].isnull().sum()

,0
Utilities,0
Exterior_1st,0
Exterior_2nd,0
Electrical,1
Kitchen_Qual,0
Functional,0
Sale_Type,0


In [353]:
df[mode_categorical].head()

,Utilities,Exterior_1st,Exterior_2nd,Electrical,Kitchen_Qual,Functional,Sale_Type
0,AllPub,BrkFace,Plywood,SBrkr,TA,Typ,WD
1,AllPub,VinylSd,VinylSd,SBrkr,TA,Typ,WD
2,AllPub,Wd Sdng,Wd Sdng,SBrkr,Gd,Typ,WD
3,AllPub,BrkFace,BrkFace,SBrkr,Ex,Typ,WD
4,AllPub,VinylSd,VinylSd,SBrkr,TA,Typ,WD


In [354]:
for col in mode_categorical:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

- **`special_case_temporal` (Impute missing indices dynamically with that row's `Year_Built` value)**

In [355]:
special_case_temporal = ['Garage_Yr_Blt']

In [356]:
df[temporal_features].isnull().sum()

,0
Year_Built,0
Year_Remod/Add,0
Garage_Yr_Blt,159
Yr_Sold,0
Mo_Sold,0


In [357]:
df[temporal_features].head()

,Year_Built,Year_Remod/Add,Garage_Yr_Blt,Yr_Sold,Mo_Sold
0,1960.0,1960,1960.0,2010,5
1,1961.0,1961,1961.0,2010,6
2,1958.0,1958,1958.0,2010,6
3,1968.0,1968,1968.0,2010,4
4,1997.0,1998,1997.0,2010,3


In [358]:
df['Garage_Yr_Blt'] = df['Garage_Yr_Blt'].fillna(df['Year_Built'])

In [359]:
df[temporal_features].isnull().sum()

,0
Year_Built,0
Year_Remod/Add,0
Garage_Yr_Blt,0
Yr_Sold,0
Mo_Sold,0


**Final check for existence of null values**

In [360]:
# Null values left?
print(df.isnull().values.any())
if df.isnull().values.any():
  print("Yes, there exist null values!")
  null_counts = df.isnull().sum()
  null_counts = null_counts[null_counts > 0]
  print(null_counts)
else:
  print("No null values")

False
No null values


> **Final dataset with no missing values**

In [361]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,113.0,17727.75,Pave,No Alley,IR1,Lvl,AllPub,Corner,...,0,No Pool,No Fence,None,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622.00,Pave,No Alley,Reg,Lvl,AllPub,Inside,...,0,No Pool,MnPrv,None,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267.00,Pave,No Alley,IR1,Lvl,AllPub,Corner,...,0,No Pool,No Fence,Gar2,0,6,2010,WD,Normal,172000
3,20,RL,93.0,11160.00,Pave,No Alley,Reg,Lvl,AllPub,Corner,...,0,No Pool,No Fence,None,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830.00,Pave,No Alley,IR1,Lvl,AllPub,Inside,...,0,No Pool,MnPrv,None,0,3,2010,WD,Normal,189900


In [362]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 80 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   MS_SubClass      2930 non-null   category
 1   MS_Zoning        2930 non-null   category
 2   Lot_Frontage     2930 non-null   float64 
 3   Lot_Area         2930 non-null   float64 
 4   Street           2930 non-null   category
 5   Alley            2930 non-null   category
 6   Lot_Shape        2930 non-null   category
 7   Land_Contour     2930 non-null   category
 8   Utilities        2930 non-null   category
 9   Lot_Config       2930 non-null   category
 10  Land_Slope       2930 non-null   category
 11  Neighborhood     2930 non-null   category
 12  Condition_1      2930 non-null   category
 13  Condition_2      2930 non-null   category
 14  Bldg_Type        2930 non-null   category
 15  House_Style      2930 non-null   category
 16  Overall_Qual     2930 non-null   category


## 4. Feature Creation / Engineering

In [363]:
import numpy as np
# New Features for Housing Dataset

# Total square footage
df['TotalSF'] = df['1st_Flr_SF'] + df['2nd_Flr_SF'] + df['Total_Bsmt_SF']

# Age of the house at time of sale
df['Age'] = df['Yr_Sold'] - df['Year_Built']

# Ensure Half_Bath and Bsmt_Half_Bath are numeric for calculations
df['Half_Bath'] = df['Half_Bath'].astype(int)
df['Bsmt_Half_Bath'] = df['Bsmt_Half_Bath'].astype(int)
df['Full_Bath'] = df['Full_Bath'].astype(int)
df['Bsmt_Full_Bath'] = df['Bsmt_Full_Bath'].astype(int)

# Total number of bathrooms
df['TotalBathrooms'] = (df['Full_Bath'] + (0.5 * df['Half_Bath']) +
                        df['Bsmt_Full_Bath'] + (0.5 * df['Bsmt_Half_Bath'])).astype('category')

# Boolean indicator for pool presence
df['HasPool'] = df['Pool_Area'].apply(lambda x: 1 if x > 0 else 0).astype('category')

# Total porch area
df['TotalPorchSF'] = df['Open_Porch_SF'] + df['Enclosed_Porch'] + df['3Ssn_Porch'] + df['Screen_Porch']

# Ensure Garage_Cars is numeric for calculations
df['Garage_Cars'] = df['Garage_Cars'].astype(int)

# Total parking capacity, considering paved drive
df['TotalParking'] = (df['Garage_Cars'] + df['Paved_Drive'].apply(lambda x:1 if x == 'Y' else 0)).astype('category')
# Boolean indicator for remodeling
df['Remodeled'] = (df['Year_Remod/Add'] != df['Year_Built']).astype(int).astype('category')
# Years since last remodel at time of sale
df['YearsSinceRemodel'] = df['Yr_Sold'] - df['Year_Remod/Add'].astype(int)

# Total number of rooms
df['TotalRooms'] = df['TotRms_AbvGrd'].astype('category') # TotRms_AbvGrd is already category

# Ratio of living area to lot area
df['LotRatio'] = df['Gr_Liv_Area'] / df['Lot_Area']

# Sale price per square foot (if SalePrice is available)
if 'SalePrice' in df.columns:
    df['PricePerSqFt'] = df['SalePrice'] / df['TotalSF']

# Categorical age group of the house
def age_group(age):
    if age <= 10:
        return 'New'
    elif age <= 30:
        return 'Modern'
    elif age <= 50:
        return 'Mature'
    else:
        return 'Old'

df['AgeGroup'] = df['Age'].apply(age_group).astype('category')

# Total outdoor living space
df['TotalOutdoorSF'] = df['Wood_Deck_SF'] + df['TotalPorchSF']

# Boolean indicator for fireplace presence
df['HasFireplace'] = df['Fireplaces'].apply(lambda x: 1 if x > 0 else 0).astype('category')

In [364]:
# Mapping for OverallQual to match ExterQual and KitchenQual ratings
overall_qual_map = {
    10: 'Ex',  # Very Excellent
    9:  'Ex',  # Excellent
    8:  'Gd',  # Very Good
    7:  'Gd',  # Good
    6:  'TA',  # Above Average
    5:  'TA',  # Average
    4:  'Fa',  # Below Average
    3:  'Fa',  # Fair
    2:  'Po',  # Poor
    1:  'Po'   # Very Poor
}

# Apply the mapping to convert OverallQual to qualitative categories
df['OverallQualCategory'] = df['Overall_Qual'].map(overall_qual_map)

# Print the updated dataframe to check the new category column
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].head())


  Overall_Qual OverallQualCategory Exter_Qual Kitchen_Qual
0            6                  TA         TA           TA
1            5                  TA         TA           TA
2            6                  TA         TA           Gd
3            7                  Gd         Gd           Ex
4            5                  TA         TA           TA


In [365]:
categorical_features_new = ['AgeGroup', 'HasPool','Remodeled', 'HasFireplace', 'TotalRooms', 'TotalParking', 'TotalBathrooms', 'OverallQualCategory']

**Change data types of newly created `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [366]:
# for col in categorical_features_new:
#     df[col] = df[col].astype('category')

In [367]:
# Mapping for ExterQual and KitchenQual
quality_map = {
    'Ex': 5,  # Excellent
    'Gd': 4,  # Good
    'TA': 3,  # Typical/Average
    'Fa': 2,  # Fair
    'Po': 1   # Poor
}

df['Exter_Qual'] = df['Exter_Qual'].map(quality_map)
df['Kitchen_Qual'] = df['Kitchen_Qual'].map(quality_map)
df['OverallQualCategory'] = df['OverallQualCategory'].map(quality_map)

In [368]:
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].head())

  Overall_Qual  OverallQualCategory Exter_Qual Kitchen_Qual
0            6                    3          3            3
1            5                    3          3            3
2            6                    3          3            4
3            7                    4          4            5
4            5                    3          3            3


In [369]:
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].dtypes)

Overall_Qual           category
OverallQualCategory       int64
Exter_Qual             category
Kitchen_Qual           category
dtype: object


In [370]:
len(df.columns.to_list())

95

In [371]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,TotalParking,Remodeled,YearsSinceRemodel,TotalRooms,LotRatio,PricePerSqFt,AgeGroup,TotalOutdoorSF,HasFireplace,OverallQualCategory
0,20,RL,113.0,17727.75,Pave,No Alley,IR1,Lvl,AllPub,Corner,...,2,0,50,7,0.093413,78.581871,Mature,272,1,3
1,20,RH,80.0,11622.00,Pave,No Alley,Reg,Lvl,AllPub,Inside,...,2,0,49,5,0.077095,59.055118,Mature,140,0,3
2,20,RL,81.0,14267.00,Pave,No Alley,IR1,Lvl,AllPub,Corner,...,2,0,52,6,0.093152,64.710309,Old,429,0,3
3,20,RL,93.0,11160.00,Pave,No Alley,Reg,Lvl,AllPub,Corner,...,3,0,42,8,0.189068,58.436115,Mature,0,1,4
4,60,RL,74.0,13830.00,Pave,No Alley,IR1,Lvl,AllPub,Inside,...,3,1,12,6,0.117787,74.266719,Modern,246,1,3


In [372]:
# Combined quality score
df['Quality_Score'] = (df['OverallQualCategory'] + df['Exter_Qual'] + df['Kitchen_Qual']) / 3

TypeError: Object with dtype category cannot perform the numpy op add

In [ ]:
df.columns.to_list()

## 5. Mathematical Transformation

## 6. Categorical Encoding

## 7. Scaling

## 8. Pipeline and ColumnTransformer

## 9. Modeling (Training)